# Training Qwen3-4B for Effect TypeScript Code Generation

Fine-tune a model to generate high-quality Effect-style TypeScript code using GRPO (Group Relative Policy Optimization).

## Hardware
- NVIDIA GeForce RTX 4090 (24GB VRAM)
- CUDA 13.0 / PyTorch 2.10.0

## Setup
```bash
uv venv unsloth_env --python 3.13
.\unsloth_env\Scripts\activate
uv pip install unsloth vllm --torch-backend=auto
```

vLLM is required for **faster inference during GRPO training** (sampling responses) and **extended context length**.

## Workflow
1. **Load model** -> Qwen3-4B with LoRA config
2. **Format data** -> Convert extracted samples to chat messages
3. **SFT pre-training** (optional) -> Teach the model the code format
4. **GRPO training** -> Reinforcement learning with reward functions
5. **Save** -> Export LoRA adapter


In [ ]:
%%capture
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # Extended context length (+30%)

print("Checking dependencies...")
try:
    import unsloth
    print(f"Unsloth {unsloth.__version__} already installed")
except ImportError:
    !pip install unsloth --quiet
    print("Unsloth installed")

try:
    import vllm
    print(f"vLLM {vllm.__version__} already installed")
except ImportError:
    !pip install vllm --quiet
    print("vLLM installed")

print("Ready.")


In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096
RANK = 64

MODEL_NAME = "unsloth/Qwen3-4B"

print(f"Loading {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=False,  # Set True when serving with vLLM
    max_lora_rank=RANK,
    gpu_memory_utilization=0.7,
)

print("Model loaded.")


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("LoRA adapters configured.")


### GRPO Chat Template

The model uses special tags for reasoning and code output:

- `<start_working_out>` / `<end_working_out>` -- reasoning
- `<CODE>` / `</CODE>` -- TypeScript code block


In [ ]:
REASONING_START = "<start_working_out>"
REASONING_END = "<end_working_out>"
CODE_START = "<CODE>"
CODE_END = "</CODE>"

SYSTEM_PROMPT = """You are an expert TypeScript developer specializing in the Effect framework.
Analyze the requirements and generate high-quality Effect code.
Provide your reasoning between <start_working_out> and <end_working_out>.
Then, provide your complete TypeScript code implementation between <CODE> and </CODE>."""

# Verify the template works
test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Generate an Effect service pattern"},
]
formatted = tokenizer.apply_chat_template(test_messages, tokenize=False)
print("=== Template test ===")
print(formatted[:300])
print("...")


### Load Training Data

428 extracted Effect TypeScript samples from `extracted-code/effect-code-samples.json`.


In [ ]:
import json
from datasets import Dataset

DATA_PATH = "extracted-code/effect-code-samples.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples")
print(f"First sample keys: {list(data[0].keys())}")


In [ ]:
def format_sample(x):
    prompt = x["prompt"]
    completion = x["completion"]
    path = x.get("path", "unknown")
    source = x.get("source", "unknown")

    reasoning = f"""Analyzing requirement: {prompt}
Source: {source}, Path: {path}
I need to generate TypeScript code using Effect framework patterns."""

    final_output = (
        REASONING_START + reasoning + REASONING_END +
        CODE_START + completion + CODE_END
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": final_output},
    ]

hf_dataset = Dataset.from_list([format_sample(s) for s in data])
hf_dataset = hf_dataset.map(lambda x: {"Messages": x["completion"]})
print(f"Formatted dataset: {len(hf_dataset)} samples")


In [ ]:
hf_dataset["token_count"] = hf_dataset["Messages"].apply(
    lambda msgs: len(tokenizer.apply_chat_template(msgs, tokenize=True))
)

print("Token count statistics:")
print(hf_dataset["token_count"].describe())

# Filter samples that fit within context window
hf_dataset = hf_dataset.filter(lambda x: x["token_count"] <= MAX_SEQ_LENGTH)
print(f"After filtering: {len(hf_dataset)} samples fit in {MAX_SEQ_LENGTH} tokens")


### Create Text Field

Convert message lists to chat template strings for training.


In [ ]:
hf_dataset["text"] = hf_dataset["Messages"].apply(
    lambda msgs: tokenizer.apply_chat_template(msgs, tokenize=False)
)

print("First 500 chars of training text:")
print(hf_dataset["text"][0][:500])


### Optional: SFT Pre-Training

Supervised fine-tuning teaches the model the code format before GRPO.

Uncomment the cells below to run SFT training.
For 428 samples, ~2 epochs takes ~10-15 minutes on RTX 4090.


In [ ]:
# from trl import SFTTrainer, SFTConfig
#
# sft_trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=hf_dataset,
#     dataset_text_field="text",
#     max_seq_length=MAX_SEQ_LENGTH,
#     args=SFTConfig(
#         per_device_train_batch_size=1,
#         gradient_accumulation_steps=4,
#         warmup_steps=5,
#         num_train_epochs=2,
#         learning_rate=2e-4,
#         fp16=not torch.cuda.is_bf16_supported(),
#         bf16=torch.cuda.is_bf16_supported(),
#         logging_steps=5,
#         optim="adamw_8bit",
#         weight_decay=0.001,
#         lr_scheduler_type="linear",
#         seed=3407,
#         output_dir="training/output",
#         report_to="none",
#     ),
# )


In [ ]:
# Uncomment to run SFT training:
# sft_trainer.train()

# Verify output format after training:
# text = tokenizer.apply_chat_template(
#     hf_dataset[0]["Messages"][:2],
#     tokenize=False,
# )
# print(text[:300])


In [ ]:
# After SFT (if run), free memory before GRPO:
# del sft_trainer
# torch.cuda.empty_cache()
# import gc
# gc.collect()


### GRPO Training

Reinforcement learning with reward functions to improve code quality.

vLLM standby mode (set `UNSLOTH_VLLM_STANDBY=1`) provides **faster inference** when the model samples responses during training.

**Rewards:**
- **+1.0** -- Code has `<CODE>` tags
- **+0.5** -- Has Effect imports
- **+0.3** -- Uses Schema
- **+0.2** -- Has exports
- **-0.5** -- Response too short (<100 chars)


In [ ]:
hf_dataset_grpo = Dataset.from_list([
    {
        "prompt": s["prompt"],
        "completion": s["completion"],
        "path": s.get("path", "unknown"),
        "source": s.get("source", "unknown"),
    }
    for s in data
])

print(f"GRPO dataset: {len(hf_dataset_grpo)} samples")
print("\nFirst sample prompt:", hf_dataset_grpo[0]["prompt"])
print("\nFirst sample completion:", hf_dataset_grpo[0]["completion"][:200])


In [ ]:
def reward_fn(completions):
    rewards = []
    for text in completions:
        score = 0.0

        # Has CODE tags
        if "<CODE>" in text and "</CODE>" in text:
            score += 1.0

        # Has Effect imports
        if "from effect" in text or "import Effect" in text:
            score += 0.5

        # Has Schema usage
        if "Schema" in text:
            score += 0.3

        # Has exports
        if "export " in text:
            score += 0.2

        # Penalize too short
        if len(text) < 100:
            score -= 0.5

        rewards.append(score)

    return torch.tensor(rewards)

# Test reward function
test_completion = "<CODE>import * as Effect from "effect"\nexport const test = Effect.succeed(42)</CODE>"
print("Test reward:", reward_fn([test_completion]).item())


In [ ]:
def format_for_grpo(x):
    prompt = x["prompt"]
    completion = x["completion"]
    path = x.get("path", "unknown")
    source = x.get("source", "unknown")

    reasoning = f"""Analyzing requirement: {prompt}
Source: {source}, Path: {path}
I need to generate TypeScript code using Effect framework patterns."""

    final_output = (
        REASONING_START + reasoning + REASONING_END +
        CODE_START + completion + CODE_END
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": final_output},
    ]

hf_dataset_grpo = hf_dataset_grpo.map(format_for_grpo)
hf_dataset_grpo = hf_dataset_grpo.map(
    lambda x: {"text": tokenizer.apply_chat_template(x["completion"], tokenize=False)}
)

print("GRPO dataset formatted.")


In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset_grpo,
    args=GRPOConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-6,
        num_train_epochs=1,
        warmup_steps=5,
        logging_steps=5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        reward_functions=[reward_fn],
        output_dir="training/output",
    ),
)

grpo_trainer.train()


### Save the Model

Export the trained LoRA adapter to `training/output/`.


In [ ]:
OUTPUT_DIR = "training/output/qwen3-4b-effect-codegen"

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to {OUTPUT_DIR}")
